
# Assignment 3: Grocery Store Management System

### Group Members
- Maheen Ahsan (25L-5080)
- Sharein Yasir (25L-5027)

## Project Overview
Our project is a console-based Grocery Store Management System. It provides two user roles: Admin and Guest Buyer.

## Features

### Admin
- Login using username and password
- Add, view, update, and remove products
- View sales reports

### Guest Buyer
- Browse and search available products
- Purchase multiple products
- Receive a final bill

## System Flow
1. The program displays the main menu with Admin Login, Guest Buyer, and Exit options.
2. Admins log in to manage inventory and view sales.
3. Guest buyers browse products and make purchases without logging in.
4. Product quantities are updated after purchases.
5. Product and sales records are stored in products.txt and sales.txt.
6. Users can return to the main menu or exit the application.

## Python Concepts Used
Functions, loops, conditional statements, lists, file handling, and exception handling.


## 1. Read and save product records
These functions read inventory from "products.txt" and save inventory changes.


In [15]:
# Load product records from the text file.
def load_products():
    products = []
    try:
        with open("products.txt", "r") as file:
            for line in file:
                parts = line.strip().split(",")
                if len(parts) == 3:
                    try:
                        products.append([parts[0], int(parts[1]), int(parts[2])])
                    except ValueError:
                        print("Skipping an invalid product record.")
    except FileNotFoundError:
        print("products.txt not found. Starting with empty inventory.")
    except PermissionError:
        print("Cannot read products.txt (permission denied).")
    return products

# Save all products after an inventory change.
def save_products(products):
    try:
        with open("products.txt", "w") as file:
            for product in products:
                file.write(product[0] + "," + str(product[1]) + "," + str(product[2]) + "\n")
        return True
    except (PermissionError, OSError):
        print("Could not save products.txt.")
        return False


## 2. Admin login
Checks the username and password before opening the admin menu.


In [16]:
# Check the administrator's credentials.
def admin_login(username, password):
    if username == "admin" and password == "1234":
        return True
    return False


## 3. Add and view products
Add new products or display and search existing products.


In [17]:
# Add a new product to the inventory.
def add_product(products):
    name = input("Product name: ").strip()
    if name == "" or "," in name:
        print("Enter a valid name without commas.")
        return
    for product in products:
        if product[0].lower() == name.lower():
            print("Product already exists. Use Update Product.")
            return
    try:
        price = int(input("Price (Rs.): "))
        quantity = int(input("Quantity: "))
        if price <= 0 or quantity < 0:
            print("Price must be positive and quantity cannot be negative.")
            return
    except ValueError:
        print("Please enter whole numbers for price and quantity.")
        return
    products.append([name, price, quantity])
    if save_products(products):
        print("Product added successfully.")

# View inventory, optionally showing only matching products.
def view_products(products, search=""):
    found = False
    print("\nName | Price (Rs.) | Quantity")
    for product in products:
        if search.lower() in product[0].lower():
            print(product[0], "|", product[1], "|", product[2])
            found = True
    if not found:
        print("No matching products found.")
    return found


## 4. Update or remove products
The admin can change product details or remove an item.


In [18]:
# Admin can update or remove an existing product.
def manage_products(products):
    name = input("Enter product name: ").strip()
    for product in products:
        if product[0].lower() == name.lower():
            print("1. Update price and quantity")
            print("2. Remove product")
            choice = input("Choice: ")
            if choice == "1":
                try:
                    price = int(input("New price: "))
                    quantity = int(input("New quantity: "))
                    if price <= 0 or quantity < 0:
                        print("Invalid price or quantity.")
                        return
                except ValueError:
                    print("Enter whole numbers only.")
                    return
                product[1] = price
                product[2] = quantity
                if save_products(products):
                    print("Product updated.")
            elif choice == "2":
                products.remove(product)
                if save_products(products):
                    print("Product removed.")
            else:
                print("Invalid choice.")
            return
    print("Product not found.")


## 5. Buy products
A guest buyer can choose products, see the bill, and confirm or cancel a purchase.


In [19]:
# Buyer can purchase several products and receive one bill.
def buy_products(products):
    cart = []
    while True:
        view_products(products)
        name = input("Product to buy (or 'done' to checkout): ").strip()
        if name.lower() == "done":
            break
        selected = None
        for product in products:
            if product[0].lower() == name.lower():
                selected = product
                break
        if selected is None:
            print("Product not found.")
            continue
        try:
            quantity = int(input("Quantity: "))
        except ValueError:
            print("Enter a whole number.")
            continue
        if quantity <= 0 or quantity > selected[2]:
            print("Invalid quantity or insufficient stock.")
            continue
        selected[2] = selected[2] - quantity
        cart.append([selected[0], quantity, selected[1] * quantity])
        print("Added to bill.")
    if len(cart) == 0:
        print("No items purchased.")
        return
    total = 0
    print("\n===== FINAL BILL =====")
    for item in cart:
        print(item[0], "x", item[1], "= Rs.", item[2])
        total = total + item[2]
    print("Total: Rs.", total)
    confirm = input("Confirm purchase? (yes/no): ").lower()
    if confirm != "yes":
        for item in cart:
            for product in products:
                if product[0] == item[0]:
                    product[2] = product[2] + item[1]
        print("Purchase cancelled.")
        return
    try:
        with open("sales.txt", "a") as file:
            for item in cart:
                file.write(item[0] + "," + str(item[1]) + "," + str(item[2]) + "\n")
    except (PermissionError, OSError):
        print("Could not record sale. Purchase cancelled.")
        for item in cart:
            for product in products:
                if product[0] == item[0]:
                    product[2] = product[2] + item[1]
        return
    if save_products(products):
        print("Purchase complete. Thank you!")
    else:
        print("Warning: Sale recorded but inventory could not be saved.")


## 6. Sales report
Reads "sales.txt" to calculate sales totals and remaining inventory.


In [20]:
# Display a simple report using saved sales.
def sales_report(products):
    total_sales = 0
    items_sold = 0
    try:
        with open("sales.txt", "r") as file:
            for line in file:
                parts = line.strip().split(",")
                if len(parts) == 3:
                    try:
                        items_sold = items_sold + int(parts[1])
                        total_sales = total_sales + int(parts[2])
                    except ValueError:
                        print("Skipping an invalid sales record.")
    except FileNotFoundError:
        print("No sales file yet.")
    except PermissionError:
        print("Cannot read sales.txt.")
        return
    stock = 0
    for product in products:
        stock = stock + product[2]
    print("\n===== SALES REPORT =====")
    print("Product types:", len(products))
    print("Units in stock:", stock)
    print("Units sold:", items_sold)
    print("Total revenue: Rs.", total_sales)


## 7. Run the interactive application
Run this cell to use the admin and buyer menus. Enter `3` in the main menu to exit.
Admin credentials for testing: username "admin", password "1234"
.


In [ ]:
# Main interactive menu.
products = load_products()
while True:
    print("\n===== GROCERY STORE =====")
    print("1. Admin Login")
    print("2. Continue as Guest (Buyer)")
    print("3. Exit")
    choice = input("Choice: ")
    if choice == "1":
        username = input("Username: ")
        password = input("Password: ")
        if not admin_login(username, password):
            print("Incorrect username or password.")
            continue
        while True:
            print("\n===== ADMIN MENU =====")
            print("1. Add Product\n2. View Products\n3. Update or Remove Product\n4. Sales Report\n5. Logout")
            admin_choice = input("Choice: ")
            if admin_choice == "1":
                add_product(products)
            elif admin_choice == "2":
                view_products(products)
            elif admin_choice == "3":
                manage_products(products)
            elif admin_choice == "4":
                sales_report(products)
            elif admin_choice == "5":
                break
            else:
                print("Invalid choice.")
    elif choice == "2":
        while True:
            print("\n===== BUYER MENU =====")
            print("1. View Products\n2. Search Product\n3. Buy Products\n4. Back")
            buyer_choice = input("Choice: ")
            if buyer_choice == "1":
                view_products(products)
            elif buyer_choice == "2":
                view_products(products, input("Search name: ").strip())
            elif buyer_choice == "3":
                buy_products(products)
            elif buyer_choice == "4":
                break
            else:
                print("Invalid choice.")
    elif choice == "3":
        print("Goodbye!")
        break
    else:
        print("Invalid choice.")



===== GROCERY STORE =====
1. Admin Login
2. Continue as Guest (Buyer)
3. Exit
Choice: 1
Username: admin
Password: 1234

===== ADMIN MENU =====
1. Add Product
2. View Products
3. Update or Remove Product
4. Sales Report
5. Logout
Choice: 2

Name | Price (Rs.) | Quantity
Milk | 250 | 10
Bread | 150 | 20
Eggs | 300 | 15
Rice | 400 | 25

===== ADMIN MENU =====
1. Add Product
2. View Products
3. Update or Remove Product
4. Sales Report
5. Logout
Choice: 1
Product name: Noodles
Price (Rs.): 250
Quantity: 10
Product added successfully.

===== ADMIN MENU =====
1. Add Product
2. View Products
3. Update or Remove Product
4. Sales Report
5. Logout
Choice: 3
Enter product name: Bread
1. Update price and quantity
2. Remove product
Choice: 1
New price: 100
New quantity: 20
Product updated.

===== ADMIN MENU =====
1. Add Product
2. View Products
3. Update or Remove Product
4. Sales Report
5. Logout
Choice: 2

Name | Price (Rs.) | Quantity
Milk | 250 | 10
Bread | 100 | 20
Eggs | 300 | 15
Rice | 400 |